# Info

This notebook allows to evaluate the performance of ASR models
- Loads a user-defined folder with speech audios
- Transcribes the audio via a remote server hosting fine-tuned models 
- Provides statistics on WER of the transcriptions
- Prepares a results file which can be further analyzed using the Nvidia Speech Explorer tool

This notebook requires the server `models-evaluator/model-server-multi.py` to be running

# ToDo

- Assess if we can use: https://github.com/NVIDIA-NeMo/NeMo/tree/stable/tools/asr_evaluator 

# Parameters

In [ ]:
# specify folders
# to benchmark transcription performance of the different ASR models

data_folders = [
    '/home/lucar/Development/chiara-speech2text_data/data_training/dataset/test'        ,
    '/home/lucar/Development/chiara-speech2text_data/data_training/dataset/train'       ,
    '/home/lucar/Development/chiara-speech2text_data/data_training/dataset/validation'
]

## Credentials

In [ ]:
# Credentials

# env file with configs and credentials
ENV_FILE = "/home/lucar/Development/chiara-speech2text_private-v2/envs/.env"

# Init notebook

In [ ]:
# Install packages

%pip install --quiet pip
%pip install --quiet huggingface_hub
%pip install --quiet torch
%pip install --quiet transformers
%pip install --quiet librosa
%pip install --quiet -U nemo_toolkit['asr']
%pip install --quiet pyaudio

In [ ]:
%pip install --upgrade --quiet lhotse

In [ ]:
# init
import os
import sys
import glob
import IPython.display as ipd

import torch
import librosa

from huggingface_hub import snapshot_download
from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError # Import HfHubHTTPError directly from utils

from transformers import WhisperForConditionalGeneration, WhisperProcessor

#from nemo.collections.asr.models import ASRModel

import requests

In [ ]:
import torch.utils.data
# Patch torch.utils.data.Sampler if it's missing __init__ or using object.__init__
# This fixes a TypeError in Lhotse/NeMo transcription with newer torch versions
if torch.utils.data.Sampler.__init__ is object.__init__:
    def sampler_init(self, data_source=None):
        pass
    torch.utils.data.Sampler.__init__ = sampler_init

In [ ]:
cuda_available = torch.cuda.is_available()
if cuda_available:
    print(f"CUDA device count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"CUDA device {i} name: {torch.cuda.get_device_name(i)}")
device = "cuda" if cuda_available else "cpu"

# cleanup
del cuda_available, i

In [ ]:
# Disable ipywidgets bars (they cause visualization errors with transforms library in notebooks)

import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]      = "1"

import datasets
datasets.disable_progress_bars()

In [ ]:
# Login to Hugging Face (to access models by "luca-r")

from huggingface_hub import login, HfApi
from huggingface_hub.utils import HfHubHTTPError

# HF login (access models by "luca-r")

# Load Hugging Face token
HF_TOKEN = os.environ.get("HF_TOKEN")
#print(f"I will use the HF token: {HF_TOKEN}")

# login to HF
login(token=HF_TOKEN) # Pass the token explicitly for consistency and clarity

# Verify login by attempting to retrieve user information
try:
    HfApi().whoami()
    print("Successfully logged in to Hugging Face Hub.")
except HfHubHTTPError as e:
    raise RuntimeError(f"Hugging Face login failed: {e}. Please ensure your 'hf_token' secret is valid and you have restarted the Colab runtime after setting it.")
except Exception as e:
    raise RuntimeError(f"An unexpected error occurred during Hugging Face login verification: {e}")

# cleanup
del HF_TOKEN

In [ ]:
# get today's date in format YYYYMMDD
# this will be used to filename the ouptut from this notebook
from datetime import datetime
string_date = datetime.now().strftime("%Y%m%d")

# Prepare results

## Define useful functions

In [ ]:
from flask import logging
import requests

# the following script serves the endpoints below
# /home/lucar/Development/chiara-speech2text_repo-private/model-server/tester_model-server.py

#
def transcribe_all(wav_file: str, base_url: str = "http://127.0.0.1:9100") -> dict:
    with open(wav_file, "rb") as f:
        files = {"wav": f}
        response = requests.post(f"{base_url}/transcribe_all", files=files)
    response.raise_for_status()
    return response.json()

## Transcribe 1 sample

In [ ]:
# select audio file
#wav_file = '/home/lucar/Development/chiara-speech2text_repo-private/assets/audio_chiara-1.wav'
#wav_file = '/home/lucar/Development/chiara-speech2text_repo-private/assets/audio_chiara-2.wav'
#wav_file = '/home/lucar/Development/chiara-speech2text_repo-private/assets/audio_chiara-3.wav'
wav_file = '/home/lucar/Development/chiara-speech2text_repo-private/assets/audio_chiara-4.wav'
#wav_file = '/home/lucar/Development/chiara-speech2text_repo-private/assets/audio_chiara-5.wav'

# show ipy widget to play audio
ipd.Audio(wav_file)

In [ ]:
response = transcribe_all(wav_file)
print(response.get("transcript_whisper_vanilla"))
print(response.get("transcript_whisper_finetuned1"))
print(response.get("transcript_whisper_finetuned2"))

In [ ]:
# cleanup
del wav_file

## Transcribe folder

In [ ]:
# Set folder for output results
DIR_RESULTS_OUTPUT = "results"

#!rm -rf {DIR_RESULTS_OUTPUT}
!mkdir  {DIR_RESULTS_OUTPUT}

In [ ]:
import json, glob, os, csv
import librosa

#
def load_json_maybe_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)  # standard JSON (list/dict)
        except json.JSONDecodeError:
            f.seek(0)
            return [json.loads(line) for line in f if line.strip()]  # JSONL

results_filename = f'{DIR_RESULTS_OUTPUT}/{string_date}_results.json'
global_count = 0

with open(results_filename, 'w') as f:
    for folder_idx, folder in enumerate(data_folders, 1):
        print(f"\n{'='*60}")
        print(f"Folder {folder_idx}/{len(data_folders)}: {folder}")
        print(f"{'='*60}")

        # ── Pre-load ground truth transcripts from .csv or .json ──
        ground_truth_map = {}

        # Try CSV first (file_name -> transcription)
        csv_files = glob.glob(os.path.join(folder, "*.csv"))
        for csv_file in csv_files:
            with open(csv_file, "r", encoding="utf-8") as cf:
                reader = csv.DictReader(cf)
                for row in reader:
                    filepath = row.get("file_name", "")
                    transcript = row.get("transcription", "")
                    if filepath and transcript:
                        ground_truth_map[os.path.basename(filepath)] = transcript

        # Also load from JSON/JSONL (audio_filepath -> text)
        json_files = glob.glob(os.path.join(folder, "*.json"))
        for json_file in json_files:
            data = load_json_maybe_jsonl(json_file)
            for entry in data:
                filepath = entry.get("audio_filepath", "")
                text = entry.get("text", "")
                if filepath and text:
                    ground_truth_map[os.path.basename(filepath)] = text

        print(f"Loaded {len(ground_truth_map)} ground truth transcripts")

        # perform inference on all audio files inside the folder
        wav_files = glob.glob(os.path.join(folder, '*.wav'))
        total_files = len(wav_files)

        for idx, wav_file in enumerate(wav_files, 1):
            #if idx > 10: break     # only process the first N files for testing

            global_count += 1
            print(f"Processing: {idx}/{total_files} - {wav_file}")

            try:
                response = transcribe_all(wav_file)
                transcript_whisper_vanilla           = response.get("transcript_whisper_vanilla")
                transcript_whisper_finetuned1        = response.get("transcript_whisper_finetuned1")
                transcript_whisper_finetuned2        = response.get("transcript_whisper_finetuned2")
                transcript_whisper_finetuned2_llm    = response.get("transcript_whisper_finetuned2_llm")

                # - read the ground truth transcript from pre-loaded map
                ground_truth = ground_truth_map.get(os.path.basename(wav_file))

                line = {
                    "audio_filepath":               wav_file,
                    "duration":                     librosa.get_duration(filename=wav_file),
                    "text":                         ground_truth,
                    "pred_text_vanilla":            transcript_whisper_vanilla,
                    "pred_text_finetuned1":         transcript_whisper_finetuned1,
                    "pred_text_finetuned2":         transcript_whisper_finetuned2,
                }
                f.write(json.dumps(line, ensure_ascii=False) + '\n')

            except Exception as e:
                print(f"Error processing {wav_file}: {e}")

print(f"\nResults saved to {results_filename} ({global_count} total files processed)")

In [ ]:
# cleanup
del f, ground_truth, line, wav_files

# Analyze results

In [ ]:
DIR_RESULTS_OUTPUT = "results"
results_filename = f'{DIR_RESULTS_OUTPUT}/results.json'

## Using Nvidia Nvidia Speech Explorer

In [ ]:
# Convert all "text" to lower-case (from compatibility with Nvidia Nvidia Speech Explorer)

import json


# Read all lines, lowercase the "text" field, and write back
with open(results_filename, "r") as f:
    lines = f.readlines()

with open(results_filename, "w") as f:
    for line in lines:
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)
        entry["text"] = entry["text"].lower()
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ Lowercased 'text' in {len(lines)} entries.")

The .jsonl file generated above can be used with Nvidia Speech Explorer tool
- https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/tools/speech_data_explorer.html 
- https://docs.nvidia.com/nemo-framework/user-guide/latest/nemotoolkit/tools/comparison_tool.html 

```bash
cd 
python3 /home/lucar/Development/nvidia-nemo/tools/speech_data_explorer/data_explorer.py         \
    /home/lucar/Development/chiara-speech2text_private-v2/models-evaluator/results/results.json \
    --names_compared pred_text_finetuned1 pred_text_finetuned2

python3 /home/lucar/Development/nvidia-nemo/tools/speech_data_explorer/data_explorer.py                     \
    /home/lucar/Development/chiara-speech2text_private-v2/models-evaluator/results/20260612_results.json    \
    --names_compared pred_text_finetuned1 pred_text_finetuned2                                              \
    --show_statistics pred_text_finetuned2
```

## Using this notebook

In [ ]:
# from the jsonl file, compute WER for each model

import json
from jiwer import wer

# read jsonl file
with open(results_filename, 'r') as f:
    lines = f.readlines()

# initialize lists to store WERs for each model
wer_whisper_vanilla         = []
wer_whisper_finetuned1      = []
wer_whisper_finetuned2      = []

# compute WER for each model and store in lists
for line in lines:
    # parse json line
    record = json.loads(line)
    ground_truth            = record['text']
    
    pred_whisper_vanilla        = record['pred_text_vanilla']
    pred_whisper_finetuned1     = record['pred_text_finetuned1']
    pred_whisper_finetuned2     = record['pred_text_finetuned2']

    # append to lists
    wer_whisper_vanilla.append          (wer(ground_truth, pred_whisper_vanilla))    
    wer_whisper_finetuned1.append       (wer(ground_truth, pred_whisper_finetuned1))
    wer_whisper_finetuned2.append       (wer(ground_truth, pred_whisper_finetuned2))

# compute average WER for each model
avg_wer_whisper_vanilla         = sum(wer_whisper_vanilla)      / len(wer_whisper_vanilla)
avg_wer_whisper_finetuned1      = sum(wer_whisper_finetuned1)       / len(wer_whisper_finetuned1)
avg_wer_whisper_finetuned2      = sum(wer_whisper_finetuned2)       / len(wer_whisper_finetuned2)

# print results
print("Average WERs of fine-tuned models:")
print(f"-    whisper_vanilla: {avg_wer_whisper_vanilla:.2f}")
print(f"- whisper_finetuned1: {avg_wer_whisper_finetuned1:.2f}")
print(f"- whisper_finetuned2: {avg_wer_whisper_finetuned2:.2f}")

# save summary to json file
summary = {
    "average_wer": {
        "whisper_vanilla":          round(avg_wer_whisper_vanilla, 2),
        "whisper_finetuned1":       round(avg_wer_whisper_finetuned1, 2),
        "whisper_finetuned2":       round(avg_wer_whisper_finetuned2, 2),
    }
}
with open(f'{DIR_RESULTS_OUTPUT}/{string_date}_results_wer.json', 'w') as f:
    json.dump(summary, f, indent=4)

In [ ]:
# convert the jsonl file to a more human readable json file
# this can be reviewed more easily by a human

import re
import json
import os

output_filename = f'{DIR_RESULTS_OUTPUT}/{string_date}_results_human-readable.json'

# remove output_filename if it already exists, to avoid confusion with previous runs
if os.path.exists(output_filename):
    os.remove(output_filename)

with open(results_filename, 'r') as infile:
    data = [json.loads(line) for line in infile]

# remove fields to make it more readable
fields_to_remove = ['audio_filepath', 'duration']

for item in data:
    for field in fields_to_remove:
        item.pop(field, None)

# save to new json file
with open(output_filename, 'w') as outfile:
    json.dump(data, outfile, indent=4, ensure_ascii=False)

# rename fields to make them shorter
with open(output_filename, 'r') as f:
    content = f.read()

# note: the extra spaces in the replacement strings are intentional, to make the columns align better when viewed in a text editor
content = re.sub(r'\btext\b',                   'ground-truth', content)
content = re.sub(r'\bpred_text_vanilla\b',      '     vanilla', content)
content = re.sub(r'\bpred_text_finetuned1\b',   '  finetuned1', content)
content = re.sub(r'\bpred_text_finetuned2\b',   '  finetuned2', content)

# save the modified content back to the file
with open(output_filename, 'w') as f:
    f.write(content)

#
print(f"Readable results saved to {output_filename}")

In [ ]:
# print histograms of average WER for each model

import matplotlib.pyplot as plt
import numpy as np

# Use dark background style
plt.style.use('dark_background')

# Create a list of model names and their average WERs
model_names = ['whisper_\nfinetuned-20260318',  'whisper_\nfinetuned-20260611']
avg_wers    = [avg_wer_whisper_finetuned1,       avg_wer_whisper_finetuned2,  ]


# Create a bar chart
plt.figure(figsize=(10, 5))

# use custom x positions with reduced spacing
x = np.arange(len(model_names)) * 0.15

# specify a narrower width for aesthetics
bars = plt.bar(x, avg_wers, color=['blue', 'orange'], width=0.1)

# Add value labels on top of bars
for bar, wer in zip(bars, avg_wers):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{wer:.2f}', ha='center', va='bottom')

plt.xlabel('Model')
plt.ylabel('Average WER')
plt.title('Average WER with respect to model-type')
plt.xticks(x, model_names, rotation=45)
plt.tight_layout()
plt.show()

# Push models to Hugging Face

In [ ]:
# push to hub
# If user wants, push model to Hugging Face Hub
# - /home/lucar/Development/chiara-speech2text/utils/9-push-model-to-hub.py

# - using git:
# -- login to HF via CLI
# python -m pip install huggingface_hub
# hf auth login (use right click to paste token, not middle-click)

# - manually through Hugging Face website
# -- https://huggingface.co/docs/hub/en/repositories-getting-started

# terminal
# git clone https://huggingface.co/username/repo_name
# e.g.: https://huggingface.co/luca-r/chiara_whisper-large-v3-turbo_20260202/tree/main?clone=true 

# - or APIs. e.g. hf_trainer.push_to_hub()

# pushing to HF can be done e.g. after evaulation and comparison against other models is done
# so that only the best user-chosen model is pushed

# Cleanup

In [ ]:
# Empty GPU RAM
import torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU RAM emptied.")
else:
    print("No GPU available, no GPU RAM to empty.")